> 在线讲义：[第 8 章 · occupancy 与延迟隐藏：一个 SM 上同时养多少个 warp](https://wikieden.github.io/cuda-to-tensor-core/lessons/08-occupancy.html) · 本书《CUDA 从线程到 Tensor Core》

# 第 8 章 · occupancy 与延迟隐藏：一个 SM 上同时养多少个 warp

本章回答：一次读显存要等几百个周期，GPU 在这段时间干什么？occupancy（占用率）是什么、怎么算？寄存器、shared memory、block 大小各怎么限制它？occupancy 是不是越高越好？

配套讲义：仓库里 `cuda-programming/notes/lessons/08-occupancy.md`（讲解、动画、练习都在讲义里）。对应原文：PG §2.3.7 Kernel Launch and Occupancy（`materials/text/cuda-programming-guide.txt` 第 4431–4557 行）、§3.2.2.2 Hardware Multithreading（第 8716–8753 行）；Best Practices §11.1–11.4（`text/cuda-c-best-practices-guide.txt` 第 2305–2474 行）；Gordić「Parallelism vs Concurrency」（`text/blog-gordic-inside-nvidia-gpus-matmul.txt` 第 75–79 行）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import math

print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 洗衣房 | 忙的比例 = 人数 × 1 ÷ 40 | `laundromat(p)` | 1 人 2.5%，40 人 100% |
| 第三步 例 1 | 2048 / 768 | `occupancy(768)` | 2 个 block，卡在线程，75% |
| 第三步 例 2 | block 数上限 32 | `occupancy(32)` | 32 个 block，卡在 block 数，50% |
| 第三步 例 3 | 228 KB / 100 KB | `occupancy(128, smem_bytes=100 * 1024)` | 2 个 block，卡在 shared，12.5% |
| 第三步 例 4 | 37 × 32 = 1184 → 1280 | `occupancy(128, regs_per_thread=37)` | 12 个 block，卡在寄存器，75% |
| 第三步 100% 的条件 | 65536 / 2048 | `LIMITS["regs"] // LIMITS["threads"]` | 每线程最多 32 个 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `LIMITS`**（来自 `ch08_occupancy.py`）

In [ ]:
LIMITS = {"threads": 2048, "blocks": 32, "regs": 65536, "smem": 233472, "max_warps": 64}

**函数 `laundromat`**（来自 `ch08_occupancy.py`）：洗衣房：一桶要洗 wash 分钟，放衣服 load 分钟；people 个人轮流用时机器有多少时间在干活。

In [ ]:
def laundromat(people: int, wash: int = 40, load: int = 1) -> float:
    """洗衣房：一桶要洗 wash 分钟，放衣服 load 分钟；people 个人轮流用时机器有多少时间在干活。"""
    return min(1.0, people * load / wash)

**函数 `regs_per_block`**（来自 `ch08_occupancy.py`）：寄存器按 warp 分配：每 warp 用 regs × 32 个，向上取整到 256 的倍数（Best Practices §11.1.1）。

In [ ]:
def regs_per_block(threads: int, regs_per_thread: int, round_to_256: bool = True) -> int:
    """寄存器按 warp 分配：每 warp 用 regs × 32 个，向上取整到 256 的倍数（Best Practices §11.1.1）。"""
    per_warp = regs_per_thread * 32
    if round_to_256:
        per_warp = math.ceil(per_warp / 256) * 256
    return math.ceil(threads / 32) * per_warp

**函数 `blocks_per_sm`**（来自 `ch08_occupancy.py`）：每道门槛各允许几个 block。

In [ ]:
def blocks_per_sm(threads: int, regs_per_thread: int = 0, smem_bytes: int = 0, round_to_256: bool = True,
                  limits: dict = LIMITS) -> dict[str, int]:
    """每道门槛各允许几个 block。"""
    big = 10**9
    r = regs_per_block(threads, regs_per_thread, round_to_256)
    return {
        "threads": limits["threads"] // threads,
        "blocks": limits["blocks"],
        "regs": limits["regs"] // r if r else big,
        "smem": limits["smem"] // smem_bytes if smem_bytes else big,
    }

**函数 `occupancy`**（来自 `ch08_occupancy.py`）：返回（每 SM 的 block 数, 卡脖子的门槛, occupancy = 驻留 warp ÷ 最多 warp）。

In [ ]:
def occupancy(threads: int, regs_per_thread: int = 0, smem_bytes: int = 0, round_to_256: bool = True,
              limits: dict = LIMITS) -> tuple[int, str, float]:
    """返回（每 SM 的 block 数, 卡脖子的门槛, occupancy = 驻留 warp ÷ 最多 warp）。"""
    gates = blocks_per_sm(threads, regs_per_thread, smem_bytes, round_to_256, limits)
    worst = min(gates, key=gates.get)
    nb = gates[worst]
    return nb, worst, nb * math.ceil(threads / 32) / limits["max_warps"]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 洗衣房：一台机器洗一桶 40 分钟，放衣服 1 分钟")
for p in (1, 10, 40):
    print(f"  {p:2d} 个人轮流用：机器忙的比例 {laundromat(p):.1%}")

In [ ]:
print("\n第三步 四个例子")
cases = [
    ("例 1 block 768 线程", {"threads": 768}),
    ("例 2 block 32 线程", {"threads": 32}),
    ("例 3 每 block 100 KB shared", {"threads": 128, "smem_bytes": 100 * 1024}),
    ("例 4 每线程 37 个寄存器", {"threads": 128, "regs_per_thread": 37}),
]
for name, kw in cases:
    nb, worst, occ = occupancy(**kw)
    print(f"  {name:<22} 每 SM {nb:2d} 个 block，卡在 {worst:<7}，occupancy = {occ:.1%}")
print("  例 4 的寄存器：37 × 32 =", 37 * 32, "→ 取整到", regs_per_block(32, 37), "个 / warp；每 block",
      regs_per_block(128, 37), "个")
print("  100% occupancy 时每线程最多", LIMITS["regs"] // LIMITS["threads"], "个寄存器")

## 3. 代码里才看得到的

1. **寄存器要按 warp 取整**：例 4 如果直接算 65536 / (37 × 128) = 13.8，会得到 13 个 block；按 warp 取整到 256 后才是 12 个。函数里的 `round_to_256` 参数可以关掉取整，对比两种算法。
2. **shared 的上限用字节**：228 KB = 233472 字节；真实硬件每个 block 还会额外占 1 KB 系统保留的 shared（PG 表里有），这里和讲义一样忽略。
3. **`occupancy` 只是理论值**：Nsight Compute 里还有"实测 occupancy"，会因为尾部效应、block 提前退出而更低（第 9 章）。

## 4. 自己改一改

- 例 4 里把每线程寄存器从 37 改成 32、40、48，看 block 数和 occupancy 怎么跳变。
- 把 `round_to_256=False` 传进例 4，看结果差在哪。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 等的时候换人干：warp 切换不花钱，多养几个 warp 轮流等；
> occupancy = 驻留 warp ÷ 最多 warp；
> 四道门槛看最紧：线程、block 数、寄存器、shared；
> 太低一定慢，太高不一定快：一个线程多件事在路上（ILP）也能藏延迟。